# Session 2, part 2: Plotting

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/t-hossein/hands-on-ml-biotech/blob/main/02_DataAndPlotting/02_Plotting.ipynb)

**Always look at your data before fitting a model.** A plot shows in a second what a table of numbers hides: outliers, long tails, groups, trends.

We use two libraries:

- **matplotlib** draws anything, one element at a time.
- **seaborn** is built on top of it and draws common statistical plots straight from a DataFrame.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

DATA = "../data/" # Or, alternatively, you can use "https://raw.githubusercontent.com/t-hossein/hands-on-ml-biotech/main/data/"


weather = pd.read_csv(DATA + "weather.csv")
weather["time"] = pd.to_datetime(weather["time"].astype(str), format="%Y%m%d%H")
weather["LUZ_wind_peak_in5h"] = weather["LUZ_wind_peak"].shift(-5)

bikes = pd.read_csv(DATA + "bikes.csv")

spam = pd.read_csv(DATA + "spam.csv").dropna()
spam["length"] = spam["text"].str.len()

mnist = pd.read_csv(DATA + "mnist_small.csv")
iris = pd.read_csv(DATA + "iris.csv")

print("weather", weather.shape,
      "\nbikes", bikes.shape, 
      "\nspam", spam.shape, 
      "\nmnist", mnist.shape, 
      "\niris", iris.shape)

---
## 1. The anatomy of a plot

`plt.subplots()` gives you two objects:

- the **figure**: the whole image
- the **axes**: one plot inside it, with its x and y axis

You draw by calling methods on the axes. **A plot without axis labels is not finished.**

First plot: the wind peak in Luzern during the first week of 2015.

In [ ]:
week = weather.iloc[:7 * 24] # 7 days

Several lines go on the same axes. Give each a `label` and call `ax.legend()`.

In [ ]:
# "Luzern" and "Bern"

## 2. Histogram: how is one variable distributed?

In [ ]:
# set an appropriate number for bins

The distribution has a **long tail** to the right: strong winds are rare. Those rare hours are exactly the ones a wind forecast should get right.

Later, we see that may have to transform *skewed* data.

## 3. Scatter plot: how are two variables related?

Does the air pressure in Luzern tell us something about the wind 5 hours later? With 25,000 points, use a small `alpha` (transparency) so that dense regions show up darker.

In [ ]:
# set s, c and alpha

Two things to see:

- The lower the pressure, the more likely a strong wind. Pressure carries **information** about the target.
- For one pressure value there are many different wind peaks. No function `wind = f(pressure)` can describe this data exactly. A model can only say what is **likely**.

## 4. Several plots in one figure

`plt.subplots(1, 2)` gives a row of two axes.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))

axes[0].scatter(weather["BAS_pressure"], weather["LUZ_pressure"], s=5, alpha=0.15)
axes[0].set_xlabel("pressure in Basel (hPa)")
axes[0].set_ylabel("pressure in Luzern (hPa)")
axes[0].set_title("Two features that move together")

axes[1].scatter(weather["LUZ_temperature"], weather["LUZ_wind_peak_in5h"], s=5, alpha=0.15)
axes[1].set_xlabel("temperature in Luzern (°C)")
axes[1].set_ylabel("wind peak 5 hours later (km/h)")
axes[1].set_title("A feature and the target")

fig.tight_layout()
plt.show()

## 5. seaborn: plots straight from a DataFrame

With seaborn you name the DataFrame and the columns, and it does the grouping, the colours and the legend. `hue` splits the data by a column.

**Bike rentals.** When do people rent bicycles? `lineplot` computes the mean of `count` for every hour.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.lineplot(data=bikes, x="hour", y="count", hue="workingday", errorbar=None, ax=ax)
ax.set_xlabel("hour of the day")
ax.set_ylabel("mean number of rentals")
ax.set_title("Working days have two rush hours, days off have one long afternoon")
ax.set_xticks(range(0, 24, 2))
plt.show()

One plot, and we know that `hour` matters, that `workingday` matters, and that their effect is **combined**: the hour means something different on a Sunday.

A **box plot** compares a number across categories. The box holds the middle half of the values, and the line inside it is the median.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
sns.boxplot(data=bikes, x="weather", y="count", order=["clear", "misty", "rain", "heavy_rain"], ax=ax)
ax.set_xlabel("weather")
ax.set_ylabel("rentals per hour")
ax.set_title("Fewer rentals in bad weather")
plt.show()

(`heavy_rain` has only 3 hours in the whole dataset, so its box means little.)

**Spam.** A count plot shows how many samples each class has. Check this for every classification dataset: if one class is rare, accuracy becomes misleading.

In [ ]:
# also see the matplotlib method
fig, ax = plt.subplots(figsize=(4, 3.5))
sns.countplot(data=spam, x="label", order=["ham", "spam"], ax=ax)
ax.set_xlabel("label")
ax.set_ylabel("number of emails")
ax.set_title("The two classes are roughly balanced")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
sns.histplot(data=spam, x="length", hue="label", hue_order=["ham", "spam"], log_scale=True, ax=ax)
ax.set_xlabel("email length (characters, log scale)")
ax.set_ylabel("number of emails")
ax.set_title("Email length by class")
plt.show()

Email lengths range from a few characters to tens of thousands, so the x axis uses a **log scale**. The two classes overlap a lot: length alone will not separate spam from ham, but it is our first numerical feature made from text.

## 6. Images are data too

**MNIST.** Each row holds one handwritten digit as 784 pixel values, plus its label. To see it, reshape the row back to 28×28 (as in the NumPy notebook) and show it with `imshow`.

In [ ]:
print(mnist.shape) 
mnist.iloc[:3, [0, 1, 300, 301, 783, 784]]

In [ ]:
pixels = mnist.drop(columns="class").to_numpy()
labels = mnist["class"].to_numpy()

fig, ax = plt.subplots(figsize=(3, 3))
ax.imshow(pixels[0].reshape(28, 28), cmap="gray")
ax.set_title(f"label: {labels[0]}")
ax.axis("off")
plt.show()

In [ ]:
# A grid of axes: 2 rows, 6 columns. axes.flat walks through all of them.


## 7. Exercises

### ✏️ Exercise 2, parts 5 and 6

Using the `iris` DataFrame:

5. Produce a scatter plot of `sepal-length` versus `sepal-width`. Make sure both axes have the correct labels.
6. Add the identity line y = x to the same plot in red.

*Hint:* for the line, create x values with `np.linspace(4, 8, 100)` and plot them against themselves.

In [ ]:
# YOUR CODE HERE

## 8. Many variables at once: the pair plot

A pair plot draws every pair of columns against each other, with a histogram of each column on the diagonal. It is a quick way to look at raw data. We take a random sample of 2000 hours to keep it fast.

In [ ]:
columns = ["BAS_pressure", "LUZ_pressure", "LUZ_wind_peak_in5h"]
sample = weather[columns].dropna().sample(2000, random_state=0)

In [ ]:
# also see scatter_matrix
grid = sns.pairplot(sample, height=2.2, plot_kws={"s": 8, "alpha": 0.3})
plt.show()

What do you see?

- `LUZ_wind_peak_in5h` has a long tail.
- At low pressure there are outliers with strong wind.
- The pressure in Basel and in Luzern are highly correlated: the second one adds little new information.

## 9. Saving a figure

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.hist(bikes["count"], bins=50)
ax.set_xlabel("rentals per hour")
ax.set_ylabel("number of hours")
ax.set_title("Bicycle rentals per hour")
fig.savefig("rentals_histogram.png") # save before plt.show()
plt.show()

## Homework

1. **Exercise 3** from the exercise sheet: for each scenario, decide classification or regression, inference or prediction, and give n and p.
2. **Exercise 2, parts 1 to 3**: look at the description of OpenML dataset 61, load it into a DataFrame and write it to a CSV file.
3. For each of the four datasets (MNIST, spam, weather, bike rentals), write down: what is one sample, what are the features, what is the target, and is it classification or regression?